### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="miami_housing",
    dataset_year="2016",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.openml.org/d/43093",
    download_description="""
We download the data from OpenML.

In this notebook, run:
    import openml

    # Load the dataset object from OpenML
    dataset = openml.datasets.get_dataset(
        43093,
        download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
        )
    dataset.get_data()[0].to_csv("../../../local-data-warehouse/miami_housing/miami_housing.csv", index=False)
""",
    # References
    academic_reference_bibtex="""@article{bourassa2021miami_housing,
  title={Big data, accessibility and urban house prices},
  author={Bourassa, Steven C and Hoesli, Martin and Merlin, Louis and Renne, John},
  journal={Urban Studies},
  volume={58},
  number={15},
  pages={3176--3195},
  year={2021},
  publisher={SAGE Publications Sage UK: London, England}
}
""",
    academic_reference_bibtex_key="bourassa2021miami_housing",
    license="CC_BY-NC-SA",
    data_tags=["IID"],
    curation_comments="""
 - 
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="SALE_PRC",
    problem_type="regression",
    objective_metric_name="rmse",
    stratify_on="SALE_PRC",
)

## Preprocessing

In [9]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/miami_housing.csv", header=0)

# Concatenate the two datasets
feature_names = [
    "LATITUDE",
    "LONGITUDE"
    "PARCELNO",         # identifier for each property
    "SALE_PRC",         # sale price ($)
    "LND_SQFOOT",       # land area (square feet) 
    "TOT_LVG_AREA",     # floor area (square feet) 
    "SPEC_FEAT_VAL",    # value of special features (e.g., swimming pools) ($)
    "RAIL_DIST",        # distance to the nearest rail line (an indicator of noise) (feet)
    "OCEAN_DIST",       # distance to the ocean (feet) 
    "WATER_DIST",       # distance to the nearest body of water (feet) 
    "CNTR_DIST",        # distance to the Miami central business district (feet)
    "SUBCNTR_DI",       # distance to the nearest subcenter (feet)
    "HWY_DIST",         #distance to the nearest highway (an indicator of noise) (feet) 
    "age",              # age of the structure
    "avno60plus",       # dummy variable for airplane noise exceeding an acceptable level 
    "month_sold",       # sale month in 2016 (1 = jan)
    "structure_quality",# quality of the structure 
]

cat_features = [
]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

In [10]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,LATITUDE,LONGITUDE,PARCELNO,SALE_PRC,LND_SQFOOT,TOT_LVG_AREA,SPEC_FEAT_VAL,RAIL_DIST,OCEAN_DIST,WATER_DIST,CNTR_DIST,SUBCNTR_DI,HWY_DIST,age,avno60plus,month_sold,structure_quality
0,25.751585,-80.354429,3.040080e+12,380000.0,8250.0,1920.0,750.0,11227.5,37268.4,7283.3,53965.6,26569.4,9971.9,55,0,12,4
1,25.539200,-80.379349,3.060190e+12,312000.0,4385.0,2857.0,1360.0,11073.6,16216.4,8882.5,105924.2,58379.5,2945.1,1,0,4,2
2,25.644376,-80.386857,3.059130e+12,330000.0,4983.0,2528.0,4136.0,902.6,28661.9,21366.4,79945.9,28959.4,444.6,14,0,10,4
3,25.878742,-80.123984,1.422350e+12,805000.0,5600.0,1767.0,3311.0,14846.0,1147.1,933.8,43390.1,32063.1,24226.2,31,0,11,4
4,25.908868,-80.184472,6.221901e+11,210000.0,7740.0,1700.0,0.0,7225.9,20624.6,4045.2,48060.4,46434.9,8336.5,41,0,6,4


## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 10,459
Columns: 24
Use sampling: False (sample size: 10,459)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['MSinceOldestTradeOpen', 'AverageMInFile', 'NetFractionInstallBurden', 'NetFractionRevolvingBurden', 'MSinceMostRecentTradeOpen', 'PercentInstallTrades', 'PercentTradesWBalance', 'NumTotalTrades', 'MSinceMostRecentDelq', 'NumSatisfactoryTrades']
Rows remaining as candidates after top-10 filter: 590 (of 10,459)

#### Duplicate Report
Total duplicate rows: 587 (5.61% of dataset)
Duplicate rows ignoring target: 588 (5.62% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,RiskPerformance,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,MaxDelq2PublicRecLast12M,MaxDelqEver,NumTotalTrades,NumTradesOpeninLast12M,PercentInstallTrades,MSinceMostRecentInqexcl7days,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance
0,Bad,69.0,148.0,4.0,66.0,41.0,0.0,0.0,100.0,-7.0,7.0,8.0,41.0,4.0,10.0,-7.0,1.0,1.0,32.0,60.0,7.0,3.0,1.0,50.0
1,Bad,77.0,229.0,3.0,109.0,23.0,0.0,0.0,100.0,-7.0,7.0,8.0,23.0,2.0,35.0,0.0,0.0,0.0,38.0,93.0,4.0,3.0,1.0,58.0
2,Bad,58.0,46.0,7.0,38.0,13.0,0.0,0.0,93.0,8.0,4.0,6.0,5.0,1.0,50.0,-7.0,2.0,2.0,80.0,84.0,5.0,4.0,1.0,90.0
3,Bad,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0,-9.0
4,Bad,80.0,226.0,2.0,66.0,35.0,0.0,0.0,100.0,-7.0,7.0,8.0,36.0,2.0,47.0,0.0,0.0,0.0,2.0,77.0,5.0,7.0,0.0,62.0


In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,RiskPerformance,category,0.0,0.0,2.0,"Bad, Good"
1,ExternalRiskEstimate,float64,0.0,0.0,61.0,"-9.0, 65.0, 66.0, 68.0, 73.0, 72.0, 70.0, 63.0, 75.0, 69.0"
2,MSinceOldestTradeOpen,float64,0.0,0.0,526.0,"-9.0, -8.0, 178.0, 132.0, 150.0, 176.0, 165.0, 183.0, 206.0, 158.0"
3,MSinceMostRecentTradeOpen,float64,0.0,0.0,112.0,"2.0, 3.0, 4.0, 5.0, 1.0, 6.0, -9.0, 7.0, 8.0, 9.0"
4,AverageMInFile,float64,0.0,0.0,237.0,"-9.0, 79.0, 71.0, 74.0, 68.0, 80.0, 75.0, 84.0, 70.0, 63.0"
5,NumSatisfactoryTrades,float64,0.0,0.0,74.0,"-9.0, 18.0, 15.0, 16.0, 22.0, 19.0, 13.0, 14.0, 21.0, 20.0"
6,NumTrades60Ever2DerogPubRec,float64,0.0,0.0,19.0,"0.0, 1.0, 2.0, -9.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0"
7,NumTrades90Ever2DerogPubRec,float64,0.0,0.0,17.0,"0.0, 1.0, -9.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 9.0"
8,PercentTradesNeverDelq,float64,0.0,0.0,72.0,"100.0, -9.0, 96.0, 97.0, 95.0, 94.0, 93.0, 92.0, 88.0, 90.0"
9,MSinceMostRecentDelq,float64,0.0,0.0,87.0,"-7.0, -9.0, 1.0, 2.0, 3.0, 4.0, 5.0, -8.0, 6.0, 8.0"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
ExternalRiskEstimate,10459.0,67.425758,21.121621,-9.0,94.0
MSinceOldestTradeOpen,10459.0,184.205373,109.683816,-9.0,803.0
MSinceMostRecentTradeOpen,10459.0,8.543455,13.301745,-9.0,383.0
AverageMInFile,10459.0,73.843293,38.782803,-9.0,383.0
NumSatisfactoryTrades,10459.0,19.428052,13.004327,-9.0,79.0
NumTrades60Ever2DerogPubRec,10459.0,0.042738,2.513910,-9.0,19.0
NumTrades90Ever2DerogPubRec,10459.0,-0.142843,2.367397,-9.0,19.0
PercentTradesNeverDelq,10459.0,86.661536,25.999584,-9.0,100.0
MSinceMostRecentDelq,10459.0,6.762406,20.501250,-9.0,83.0
MaxDelq2PublicRecLast12M,10459.0,4.928291,3.756275,-9.0,9.0


In [8]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column          rank                    
RiskPerformance 1      Bad   5459  52.19
                2     Good   5000  47.81

In [9]:
# Target Distribution
target_df

,count,pct
RiskPerformance,,
Bad,5459,52.19
Good,5000,47.81


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019d1f6d-5136-7eb1-a3c7-8ea1f7f3a353
d8791e6c64520b943f5a8fbcabe3cb2d70e39c70fd7146717d5674374a8d5b4f
